# conditional-hparam-branch — worked example 1: Skip the velocity update when momentum is zero

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conditional-hparam-branch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

An SGD step with momentum maintains a velocity buffer: `v = momentum * v + grad`, then steps with `v`. When `momentum == 0` there is no buffer to accumulate, so the step must collapse to plain `param - lr * grad`. Gating the velocity math behind `if momentum != 0` is what makes momentum-less SGD behave **exactly** like vanilla gradient descent.

## Worked solution

1. **Write the function signature.** `sgd_step(param, grad, velocity, lr, momentum)` returns the updated parameter and the (possibly unchanged) velocity buffer.
2. **Branch on the hyperparameter, not on the data.** `if momentum != 0:` we run the buffer recurrence `velocity = momentum * velocity + grad` and use that as the update.
3. **The zero path skips the buffer entirely.** In the `else` branch the update is just `grad`. We never touch `velocity`, so it stays at its initial value — this is the key to the test that momentum-less SGD equals vanilla GD.
4. **Apply the step uniformly.** Both branches converge to `new_param = param - lr * update`. Only the definition of `update` differs.
5. **Why this matters:** if you always ran the recurrence, `momentum=0` would still give `v = grad` (correct here) but in general fused/optimized code skips the read-modify-write of the buffer when the hparam is zero, both for speed and to guarantee bit-identical vanilla-GD behavior in tests.

In [ ]:
def sgd_step(param, grad, velocity, lr, momentum):
    if momentum != 0:
        velocity = momentum * velocity + grad
        update = velocity
    else:
        update = grad
    new_param = param - lr * update
    return new_param, velocity

t.manual_seed(0)
param = t.tensor([1.0, 2.0, 3.0])
grad = t.tensor([0.1, 0.2, 0.3])
vel = t.zeros(3)

p_plain, v_plain = sgd_step(param, grad, vel, lr=0.1, momentum=0.0)
p_mom, v_mom = sgd_step(param, grad, vel, lr=0.1, momentum=0.9)
print('momentum=0  param:', p_plain.tolist(), 'velocity:', v_plain.tolist())
print('momentum=0.9 param:', p_mom.tolist(), 'velocity:', v_mom.tolist())